In [16]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [17]:
df=pd.read_csv('IMDB.csv')

In [18]:
df=df.sample(500)
df.to_csv('data.csv',index=False)


In [19]:
df.head()

,review,sentiment
950,This happens to be one of my favorite horror f...,positive
239,"Anyone that is looking for an episode of ""Law ...",positive
769,Can I just start by saying I'm a fan of bad mo...,negative
192,An Insomniac's Nightmare is the story of a man...,positive
83,Doug McClure has starred in a few of these Bri...,negative


In [20]:
import nltk; nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [22]:
# data preprocessing function
def lemmatization(text):
    """Lemmatize the text"""
    lemmatizer =WordNetLemmatizer()
    text=text.split()
    text =[lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """remove stop words"""
    stop_words=set(stopwords.words("english"))
    text=[word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """remove numbers from the text"""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """convert text to lower case"""
    text=text.split()
    text= [word.lower() for word in text]
    return " ".join(text)

def removing_punctuation(text):
    """remove punctuation from text"""
    text=re.sub('[%s]' % re.escape(string.punctuation), ' ',text)
    text=text.replace(':',"")
    text=re.sub('\s+',' ',text).strip()
    return text

def removing_urls(text):
    """remove urls from the text"""
    url_pattern = re.compile(r'https?://\s+|www\.\s+')
    return url_pattern.sub(r'',text)

def normalize_text(df):
    """normalize the text data"""
    try:
        df['review']=df['review'].apply(lower_case)
        df['review']=df['review'].apply(remove_stop_words)
        df['review']=df['review'].apply(removing_numbers)
        df['review']=df['review'].apply(removing_punctuation)
        df['review']=df['review'].apply(removing_urls)
        df['review']=df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'error during the text normalization: {e}')
        raise



In [23]:
df=normalize_text(df)
df.head()

,review,sentiment
950,happens one favorite horror film rich classy p...,positive
239,anyone looking episode law order csi would loo...,positive
769,start saying fan bad movie really bad movie st...,negative
192,insomniac s nightmare story man s plunge insan...,positive
83,doug mcclure starred british produced genre ad...,negative


In [24]:
df['sentiment'].value_counts()

sentiment
negative    269
positive    231
Name: count, dtype: int64

In [25]:
x=df['sentiment'].isin(['positive','negative'])
df=df[x]

In [26]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})
df.head()

,review,sentiment
950,happens one favorite horror film rich classy p...,1
239,anyone looking episode law order csi would loo...,1
769,start saying fan bad movie really bad movie st...,0
192,insomniac s nightmare story man s plunge insan...,1
83,doug mcclure starred british produced genre ad...,0


In [27]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [28]:
vectorizer=CountVectorizer(max_features=100)
x=vectorizer.fit_transform(df['review'])
y=df['sentiment']

In [29]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.25,random_state=42)

In [14]:
import dagshub
mlflow.set_tracking_uri(' https://dagshub.com/raushankumar733329/MLOPS-CAPSTONE-PROJECT.mlflow')
dagshub.init(repo_owner='raushankumar733329',repo_name='MLOPS-CAPSTONE-PROJECT',mlflow=True)
mlflow.set_experiment('Logistic Regression Baseline')

Accessing as raushankumar733329

Initialized MLflow to track repo "raushankumar733329/MLOPS-CAPSTONE-PROJECT"

Repository raushankumar733329/MLOPS-CAPSTONE-PROJECT initialized!

<Experiment: artifact_location='mlflow-artifacts:/33dae99d41834807894d60bfef90d7a5', creation_time=1781782928619, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781782928619, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [30]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

# configure logging
logging.basicConfig(level=logging.INFO,format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("starting mlflow run...")
with mlflow.start_run():
    start_time=time.time()

    try:
        logging.info("logging preprocessing parameters...")
        mlflow.log_param('vectorizer',"Bag of words")
        mlflow.log_param("num_features",100)
        mlflow.log_param("test_size",0.25)

        logging.info("initiallizing logistic regression model...")
        model=LogisticRegression(max_iter=1000)

        logging.info("fitting the model")
        model.fit(x_train,y_train)
        logging.info("model training complete")

        logging.info("logging model parameter")
        mlflow.log_param("model","Logistic Regression")

        logging.info("making predictions..")
        y_pred = model.predict(x_test)

        logging.info("calculating evaluation metrics")
        accuracy=accuracy_score(y_test,y_pred)
        precision=precision_score(y_test,y_pred)
        recall=recall_score(y_test,y_pred)
        f1=f1_score(y_test,y_pred)

        logging.info("logging evaluation metrics")
        mlflow.log_metric("accuracy",accuracy)
        mlflow.log_metric("precision",precision)
        mlflow.log_metric("recall",recall)
        mlflow.log_metric("f1_score",f1)

        logging.info("saving and logging the model")
        mlflow.sklearn.log_model(model,"model")

        #  log execution time
        end_time=time.time()
        logging.info(f"model training and logging completed in {end_time - start_time:.2f} seconds.")

        # print the result for verififcation
        logging.info(f"accuracy: {accuracy}")
        logging.info(f"precision: {precision}")
        logging.info(f"recall: {recall}")
        logging.info(f"f1_score: {f1}")

    except Exception as e:
        logging.error(f"an error occured: {e}",exc_info=True)
        

2026-06-19 10:15:37,715 - INFO - starting mlflow run...
2026-06-19 10:15:39,526 - INFO - logging preprocessing parameters...
2026-06-19 10:15:40,854 - INFO - initiallizing logistic regression model...
2026-06-19 10:15:40,854 - INFO - fitting the model
2026-06-19 10:15:40,884 - INFO - model training complete
2026-06-19 10:15:40,884 - INFO - logging model parameter
2026-06-19 10:15:41,471 - INFO - making predictions..
2026-06-19 10:15:41,471 - INFO - calculating evaluation metrics
2026-06-19 10:15:41,513 - INFO - logging evaluation metrics
2026-06-19 10:15:43,323 - INFO - saving and logging the model
2026/06/19 10:15:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-06-19 10:16:06,164 - INFO - model training and logging completed in 26.64 seconds.
2026-06-19 10:16:06,164 - INFO - accuracy: 0.664
2026-06-19 10:16:06,164 - INFO - precision: 0.6949152542372882
2026-06-19 10:16:06,169 - INFO - recall: 0.6307692307692307
2026-06-19 10:16:06,169 - I

🏃 View run luminous-bear-766 at:  https://dagshub.com/raushankumar733329/MLOPS-CAPSTONE-PROJECT.mlflow/#/experiments/0/runs/8d26652623054de4837825f01d7e2c8a
🧪 View experiment at:  https://dagshub.com/raushankumar733329/MLOPS-CAPSTONE-PROJECT.mlflow/#/experiments/0
